In [1]:
import numpy as np
import pandas as pd
from scipy.stats import t, norm
from scipy.stats import linregress

In [2]:
# function for fitting yexp = intercept + slope * ycalc
# scipy.stats.linregress takes inputs as (x, y)
# we will set x = exp, y = calc
def linear_scale(calc, exp):
    slope, intercept, r_value, p_value, stderr = linregress(exp, calc)
    scaled_calc = (calc - intercept) / slope
    return scaled_calc, slope, intercept, r_value

In [3]:
# load 13C shifts from calculated values of structures A and B, and experimental values
df = pd.read_csv("C.csv")

calc_A = df["calcA"].values.astype(float)
calc_B = df["calcB"].values.astype(float)
exp    = df["exp"].values.astype(float)

In [4]:
MAE_A=np.mean(np.abs(calc_A-exp))
SDE_A=np.std(calc_A-exp)

MAE_B=np.mean(np.abs(calc_B-exp))
SDE_B=np.std(calc_B-exp)

print(f"A: mean absolute error: {MAE_A:.3f}, standard deviation = {SDE_A:.3f}")
print(f"B: mean absolute error: {MAE_B:.3f}, standard deviation = {SDE_B:.3f}")

A: mean absolute error: 1.505, standard deviation = 1.591
B: mean absolute error: 1.622, standard deviation = 1.831


In [5]:
scaled_A, slope_A, intercept_A, rA = linear_scale(calc_A, exp)
scaled_B, slope_B, intercept_B, rB = linear_scale(calc_B, exp)

print(f"A: slope = {slope_A:.3f}, intercept = {intercept_A:.3f}, r = {rA:.4f}")
print(f"B: slope = {slope_B:.3f}, intercept = {intercept_B:.3f}, r = {rB:.4f}")

A: slope = 0.927, intercept = 3.362, r = 0.9932
B: slope = 0.925, intercept = 3.646, r = 0.9901


In [6]:
error_A = scaled_A - exp
error_B = scaled_B - exp

muA=0; sigma_A=2.306
muB=0; sigma_B=2.306

tA=np.abs(error_A-muA)/sigma_A
tB=np.abs(error_B-muB)/sigma_B

nu=11.38 # degree of freedom

cdfA=t.cdf(tA, nu)
cdfB=t.cdf(tB, nu)

prod_cdfA=np.product(1-cdfA)
prod_cdfB=np.product(1-cdfB)

DP4_A=prod_cdfA/(prod_cdfA+prod_cdfB)
DP4_B=prod_cdfB/(prod_cdfA+prod_cdfB)

print(f"A: DP4: {DP4_A:.3f}")
print(f"B: DP4: {DP4_B:.3f}")

A: DP4: 0.903
B: DP4: 0.097
